# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The research paper reports that the proposed model improves content prioritization compared to the baseline.

### My Methodology Question

How was the target label created? Was it based on an observed future outcome, or was it derived from existing rules? Using observed outcomes reduces the risk of learning a predefined rule instead of real patterns.

---

## Finding 2

The paper reports improved model performance using grouped validation.

### My Methodology Question

Was the validation performed using grouped or time-aware splits? This is important because random splits may introduce information leakage and overestimate model performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# -----------------------------
# Locate repository
# -----------------------------
if os.path.exists("/content/sanju"):
    os.chdir("/content/sanju")

print("Current Directory:", os.getcwd())

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

# -----------------------------
# Create Opportunity Score
# (Higher impressions + Lower CTR)
# -----------------------------
df["opportunity_score"] = (
    df["impressions_90d"] * (1 - df["ctr"] / 100)
)

# -----------------------------
# Features
# -----------------------------
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "engagement_rate"
]

X = df[features]
y = df["opportunity_score"]

# -----------------------------
# Grouped Split
# -----------------------------
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# -----------------------------
# Train Random Forest
# -----------------------------
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------
rf_mae = mean_absolute_error(y_test, predictions)

print("\nRandom Forest MAE:", round(rf_mae, 3))

# -----------------------------
# Baseline Comparison
# -----------------------------
baseline_mae = 0.65   # Replace with your Week-4 baseline if available

comparison = pd.DataFrame({
    "Model": ["Week 4 Baseline", "Random Forest"],
    "MAE": [baseline_mae, rf_mae]
})

print("\nModel Comparison")
display(comparison)

Current Directory: /content/sanju
Dataset Shape: (30000, 44)

Random Forest MAE: 29.942

Model Comparison


,Model,MAE
0,Week 4 Baseline,0.650000
1,Random Forest,29.941838


## Honest Validation

The Week 5 model was evaluated using a grouped split based on client_id. This prevents pages from the same client appearing in both training and testing datasets.

Using grouped validation provides a more realistic estimate of model performance compared to a random split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The model was reviewed for possible data leakage.

The following columns were deliberately excluded:

- trend_direction
- trend_pct

These fields are derived from future trend information and should not be used as input features.

Identifier columns such as client_id and content_id were used only for grouping and were not included as predictive features.

The selected features represent information available before making a recommendation.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

Original Claim

The Random Forest model accurately identifies all pages requiring refresh.

Rewritten Claim

The Random Forest model identified patterns associated with content refresh opportunities on the available dataset. The results should be interpreted as decision-support rather than proof that refreshing a page will improve future search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.